In [0]:

# DAY 2 - SILVER TRANSFORMATIONS

# Load Bronze tables
bls_bronze = spark.table(
    "workspace.rearc.bronze_bls_data"
)

series_bronze = spark.table(
    "workspace.rearc.bronze_bls_series"
)

datausa_bronze = spark.table(
    "workspace.rearc.bronze_datausa_population"
)

# Inspect schemas
print("BLS OBSERVATION SCHEMA")
bls_bronze.printSchema()

print("\nBLS SERIES SCHEMA")
series_bronze.printSchema()

print("\nDATAUSA POPULATION SCHEMA")
datausa_bronze.printSchema()

print("\nBRONZE TABLES LOADED SUCCESSFULLY")


In [0]:

from pyspark.sql.functions import (
    col,
    trim,
    expr,
    current_timestamp
)

# Read Bronze BLS observations
bls_bronze = spark.table(
    "workspace.rearc.bronze_bls_data"
)

# Standardize and convert data types
bls_clean = (
    bls_bronze
    .withColumn("series_id", trim(col("series_id")))
    .withColumn("period", trim(col("period")))
    .withColumn("year", expr("try_cast(trim(year) AS INT)"))
    .withColumn("value", expr("try_cast(trim(value) AS DOUBLE)"))
)

# Keep records with valid required fields
bls_valid = bls_clean.filter(
    col("series_id").isNotNull() &
    (col("series_id") != "") &
    col("year").isNotNull() &
    col("period").isin("Q01", "Q02", "Q03", "Q04", "Q05") &
    col("value").isNotNull()
)

# Count rejected records
source_count = bls_bronze.count()
valid_count = bls_valid.count()
rejected_count = source_count - valid_count

print("BLS SILVER DATA QUALITY")
print("-----------------------")
print("Bronze records:", source_count)
print("Valid records:", valid_count)
print("Rejected records:", rejected_count)

# Add Silver processing timestamp
silver_bls_df = bls_valid.withColumn(
    "_silver_processed_timestamp",
    current_timestamp()
)

# Save Silver Delta table
(
    silver_bls_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.rearc.silver_bls_data")
)

print("\nSILVER BLS TABLE CREATED")

display(silver_bls_df.limit(10))


In [0]:

from pyspark.sql.functions import col, current_timestamp

# Load Bronze reference tables
series = spark.table("workspace.rearc.bronze_bls_series").alias("s")
sector = spark.table("workspace.rearc.bronze_bls_sector").alias("sec")
class_df = spark.table("workspace.rearc.bronze_bls_class").alias("cl")
measure = spark.table("workspace.rearc.bronze_bls_measure").alias("m")
duration = spark.table("workspace.rearc.bronze_bls_duration").alias("d")
seasonal = spark.table("workspace.rearc.bronze_bls_seasonal").alias("se")

# Join series with descriptive metadata
silver_series_df = (
    series
    .join(sector, col("s.sector_code") == col("sec.sector_code"), "left")
    .join(class_df, col("s.class_code") == col("cl.class_code"), "left")
    .join(measure, col("s.measure_code") == col("m.measure_code"), "left")
    .join(duration, col("s.duration_code") == col("d.duration_code"), "left")
    .join(seasonal, col("s.seasonal") == col("se.seasonal_code"), "left")
    .select(
        col("s.series_id"),
        col("s.sector_code"),
        col("sec.sector_name"),
        col("s.class_code"),
        col("cl.class_text"),
        col("s.measure_code"),
        col("m.measure_text"),
        col("s.duration_code"),
        col("d.duration_text"),
        col("s.seasonal"),
        col("se.seasonal_text"),
        col("s.base_year"),
        col("s.begin_year"),
        col("s.begin_period"),
        col("s.end_year"),
        col("s.end_period")
    )
    .withColumn("_silver_processed_timestamp", current_timestamp())
)

# Save Silver metadata table
(
    silver_series_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.rearc.silver_bls_series")
)

print("SILVER BLS SERIES CREATED")
print("Total series:", silver_series_df.count())

display(silver_series_df.limit(10))


In [0]:

from pyspark.sql.functions import (
    col,
    trim,
    current_timestamp
)

# Load DataUSA Bronze
datausa_bronze = spark.table(
    "workspace.rearc.bronze_datausa_population"
)

# Clean and convert data types
silver_datausa_df = (
    datausa_bronze
    .withColumn("nation_id", trim(col("nation_id")))
    .withColumn("nation", trim(col("nation")))
    .withColumn("year", col("year").cast("int"))
    .withColumn("population", col("population").cast("double"))
)

# Apply data quality rules
silver_datausa_df = silver_datausa_df.filter(
    (col("nation") == "United States") &
    col("year").isNotNull() &
    col("population").isNotNull() &
    (col("population") > 0)
)

# Add processing timestamp
silver_datausa_df = silver_datausa_df.withColumn(
    "_silver_processed_timestamp",
    current_timestamp()
)

# Save Silver Delta table
(
    silver_datausa_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.rearc.silver_datausa_population")
)

print("DATAUSA SILVER TABLE CREATED")
print("Total records:", silver_datausa_df.count())

display(silver_datausa_df.orderBy("year"))


In [0]:

# FINAL SILVER LAYER VALIDATION

bls = spark.table("workspace.rearc.silver_bls_data")
series = spark.table("workspace.rearc.silver_bls_series")
population = spark.table(
    "workspace.rearc.silver_datausa_population"
)

print("SILVER LAYER VALIDATION")
print("-" * 45)

print("BLS observations:", bls.count())
print("BLS series:", series.count())
print("Population records:", population.count())

# Check duplicate series IDs
duplicate_series = (
    series.groupBy("series_id")
    .count()
    .filter("count > 1")
    .count()
)

# Check duplicate population years
duplicate_population_years = (
    population.groupBy("year")
    .count()
    .filter("count > 1")
    .count()
)

# Check numeric data types
print("\nBLS year type:", bls.schema["year"].dataType)
print("BLS value type:", bls.schema["value"].dataType)
print("Population type:", population.schema["population"].dataType)

print("\nDuplicate series IDs:", duplicate_series)
print("Duplicate population years:", duplicate_population_years)

assert bls.count() == 77126
assert series.count() == 282
assert population.count() == 11
assert duplicate_series == 0
assert duplicate_population_years == 0

print("\nSILVER VALIDATION PASSED")
